In [14]:
import os
import glob
import json
import zipfile
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ---------------------------------------------------------
# 1. CẤU HÌNH THƯ MỤC & MÔ HÌNH
# ---------------------------------------------------------
DISEASE_CSV_PATH = "/kaggle/input/datasets/phucthaiv02/disease/disease.csv"
INPUT_DIR = "/kaggle/input/datasets/phucthaiv02/predicted/output"
OUTPUT_DIR = "/kaggle/working/output_updated"
ZIP_OUTPUT_PATH = "/kaggle/working/output_updated.zip"

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR"
TOP_K = 1  # Số lượng candidates cần lấy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print(f"[DEBUG 1] Thiết bị sử dụng: {device}")
print("=" * 60)

# ---------------------------------------------------------
# 2. TẢI MÔ HÌNH VÀ CƠ SỞ DỮ LIỆU TÊN BỆNH
# ---------------------------------------------------------
print("\n[DEBUG 2] Đang tải Model và Tokenizer SapBERT...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()


[DEBUG 1] Thiết bị sử dụng: cuda

[DEBUG 2] Đang tải Model và Tokenizer SapBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

In [15]:
print(f"[DEBUG 2] Đang đọc file CSV từ: {DISEASE_CSV_PATH}")
df_disease = pd.read_csv(DISEASE_CSV_PATH)
disease_list = df_disease["name"].astype(str).tolist()
disease_code = df_disease["code"].astype(str).tolist()

print(f"-> Tổng số tên bệnh đọc được: {len(disease_list)}")
print(f"-> Ví dụ 3 bệnh đầu tiên: {disease_list[:3]}")
print(f"-> Ví dụ 3 code đầu tiên: {disease_code[:3]}")

[DEBUG 2] Đang đọc file CSV từ: /kaggle/input/datasets/phucthaiv02/disease/disease.csv
-> Tổng số tên bệnh đọc được: 13479
-> Ví dụ 3 bệnh đầu tiên: ['Sững sờ', 'Hôn mê, không đặc hiệu', 'Buồn ngủ']
-> Ví dụ 3 code đầu tiên: ['R40.1', 'R40.2', 'R40.0']


In [16]:
# ---------------------------------------------------------
# 3. HÀM TẠO EMBEDDINGS
# ---------------------------------------------------------
def get_sapbert_embeddings(texts, batch_size=128):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts, 
            padding=True, 
            truncation=True, 
            max_length=64, 
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            cls_embeds = outputs.last_hidden_state[:, 0, :]
            norm_embeds = F.normalize(cls_embeds, p=2, dim=1)
            all_embeddings.append(norm_embeds)
            
    return torch.cat(all_embeddings, dim=0)

# Encode danh sách tên bệnh
print("\n" + "=" * 60)
print("[DEBUG 3] Đang tạo Embeddings cho toàn bộ tên bệnh trong CSV...")
disease_embeddings = get_sapbert_embeddings(disease_list, batch_size=128)
print(f"-> Hoàn tất! Shape của disease_embeddings: {disease_embeddings.shape}")
print("=" * 60)


[DEBUG 3] Đang tạo Embeddings cho toàn bộ tên bệnh trong CSV...
-> Hoàn tất! Shape của disease_embeddings: torch.Size([13479, 768])


In [17]:
# ---------------------------------------------------------
# 4. DUYỆT VÀ XỬ LÝ TẤT CẢ FILE JSON
# ---------------------------------------------------------
json_files = glob.glob(os.path.join(INPUT_DIR, "*.json"))
print(f"\n[DEBUG 4] Tìm thấy {len(json_files)} file JSON trong thư mục đầu vào.\n")

for f_idx, file_path in enumerate(json_files, 1):
    file_name = os.path.basename(file_path)
    output_file_path = os.path.join(OUTPUT_DIR, file_name)
    
    print("-" * 50)
    print(f"[{f_idx}/{len(json_files)}] Đang xử lý file: {file_name}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            json_data = json.load(f)
        except Exception as e:
            print(f"   [LỖI] Không thể đọc file: {e}")
            continue

    # Lọc các entity kiểu "CHẨN_ĐOÁN"
    chan_doan_indices = [
        i for i, item in enumerate(json_data) 
        if isinstance(item, dict) and item.get("type") == "CHẨN_ĐOÁN"
    ]
    queries = [json_data[i]["text"] for i in chan_doan_indices]

    print(f"   -> Tìm thấy {len(queries)} entity kiểu 'CHẨN_ĐOÁN'")

    if queries:
        query_embeddings = get_sapbert_embeddings(queries, batch_size=32)
        similarity_matrix = torch.mm(query_embeddings, disease_embeddings.T)
        top_scores, top_indices = torch.topk(similarity_matrix, k=TOP_K, dim=1)
        
        # Cập nhật và PRINT DEBUG chi tiết từng query
        for q_idx, original_json_idx in enumerate(chan_doan_indices):
            query_text = json_data[original_json_idx]["text"]
            old_candidates = json_data[original_json_idx]["candidates"]
            matched_indices = top_indices[q_idx].tolist()
            matched_scores = top_scores[q_idx].tolist()
            matched_candidates = [disease_code[idx] for idx in matched_indices]
            matched_names = [disease_list[idx] for idx in matched_indices]
            
            # Cập nhật JSON
            json_data[original_json_idx]["candidates"] = matched_candidates
            
            # Print debug kết quả tìm kiếm
            print(f"\n      • Query [{q_idx + 1}/{len(queries)}]: '{query_text}'")
            print(f"Old cands: {old_candidates}")
            for rank, (cand, name, score) in enumerate(zip(matched_candidates, matched_names, matched_scores), 1):
                print(f"        Top {rank}: {cand:<40} {name:<40} (Score: {score:.4f})")

    # Ghi file JSON đã cập nhật
    with open(output_file_path, "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n   ✓ Đã ghi file kết quả: {output_file_path}")




[DEBUG 4] Tìm thấy 100 file JSON trong thư mục đầu vào.

--------------------------------------------------
[1/100] Đang xử lý file: 67.json
   -> Tìm thấy 5 entity kiểu 'CHẨN_ĐOÁN'

      • Query [1/5]: 'suy tim'
Old cands: []
        Top 1: I51.7                                    Tim to                                   (Score: 0.7685)

      • Query [2/5]: 'bệnh mạch máu ngoại biên'
Old cands: []
        Top 1: I73.9                                    Bệnh mạch máu ngoại biên, không đặc hiệu (Score: 0.9436)

      • Query [3/5]: 'bệnh phổi tắc nghẽn mạn tính'
Old cands: []
        Top 1: J44.9                                    Bệnh phổi tắc nghẽn mạn tính, không xác định (Score: 0.9385)

      • Query [4/5]: 'Ngưng thở khi ngủ do tắc nghẽn'
Old cands: []
        Top 1: G47.3                                    Ngừng thở khi ngủ                        (Score: 0.8673)

      • Query [5/5]: 'Ung thư biểu mô tế bào vảy xâm nhập của dương vậtbiệt hóa kém'
Old cands: ['C76.81']
        

In [18]:
# ---------------------------------------------------------
# 5. ZIP TOÀN BỘ FILE KẾT QUẢ VÀO 1 FILE .ZIP
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("[DEBUG 5] Đang nén các file JSON kết quả vào 1 file ZIP...")
zipped_count = 0

with zipfile.ZipFile(ZIP_OUTPUT_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, OUTPUT_DIR)
            zipf.write(file_path, arcname)
            zipped_count += 1
            print(f"   + Đã thêm vào ZIP: {arcname}")

print("=" * 60)
print(f"[HOÀN THÀNH] Đã zip thành công {zipped_count} file!")
print(f"Đường dẫn file ZIP cuối cùng: {ZIP_OUTPUT_PATH}")
print("=" * 60)


[DEBUG 5] Đang nén các file JSON kết quả vào 1 file ZIP...
   + Đã thêm vào ZIP: 98.json
   + Đã thêm vào ZIP: 37.json
   + Đã thêm vào ZIP: 53.json
   + Đã thêm vào ZIP: 35.json
   + Đã thêm vào ZIP: 85.json
   + Đã thêm vào ZIP: 96.json
   + Đã thêm vào ZIP: 69.json
   + Đã thêm vào ZIP: 65.json
   + Đã thêm vào ZIP: 82.json
   + Đã thêm vào ZIP: 31.json
   + Đã thêm vào ZIP: 55.json
   + Đã thêm vào ZIP: 77.json
   + Đã thêm vào ZIP: 30.json
   + Đã thêm vào ZIP: 54.json
   + Đã thêm vào ZIP: 49.json
   + Đã thêm vào ZIP: 46.json
   + Đã thêm vào ZIP: 39.json
   + Đã thêm vào ZIP: 84.json
   + Đã thêm vào ZIP: 60.json
   + Đã thêm vào ZIP: 61.json
   + Đã thêm vào ZIP: 88.json
   + Đã thêm vào ZIP: 21.json
   + Đã thêm vào ZIP: 15.json
   + Đã thêm vào ZIP: 73.json
   + Đã thêm vào ZIP: 86.json
   + Đã thêm vào ZIP: 74.json
   + Đã thêm vào ZIP: 97.json
   + Đã thêm vào ZIP: 25.json
   + Đã thêm vào ZIP: 20.json
   + Đã thêm vào ZIP: 48.json
   + Đã thêm vào ZIP: 12.json
   + Đã th

In [19]:
import os
import glob
import json

# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN THƯ MỤC
# ---------------------------------------------------------
INPUT_DIR = "/kaggle/input/notebooks/phucthaiv02/vllm-inference/output"
OUTPUT_DIR = "/kaggle/working/output_updated"

# Lấy danh sách các file JSON
input_files = glob.glob(os.path.join(INPUT_DIR, "*.json"))

print("=" * 80)
print(f"TÌM THẤY {len(input_files)} FILE JSON ĐỂ SO SÁNH")
print("=" * 80)

for f_idx, input_path in enumerate(input_files[:2], 1):
    file_name = os.path.basename(input_path)
    output_path = os.path.join(OUTPUT_DIR, file_name)
    
    print("\n" + "#" * 80)
    print(f"FILE [{f_idx}/{len(input_files)}]: {file_name}")
    print("#" * 80)
    
    # 1. Đọc file GỐC (Trước khi thay đổi)
    if os.path.exists(input_path):
        with open(input_path, "r", encoding="utf-8") as f:
            data_before = json.load(f)
        print("\n🔴 --- [TRƯỚC KHI THAY ĐỔI (GỐC)] ---")
        print(json.dumps(data_before, ensure_ascii=False, indent=2))
    else:
        print(f"\n🔴 Không tìm thấy file gốc tại: {input_path}")
        
    # 2. Đọc file ĐÃ XỬ LÝ (Sau khi thay đổi)
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            data_after = json.load(f)
        print("\n🟢 --- [SAU KHI THAY ĐỔI (ĐÃ CẬP NHẬT CANDIDATES)] ---")
        print(json.dumps(data_after, ensure_ascii=False, indent=2))
    else:
        print(f"\n🟢 Chưa tìm thấy file đầu ra tại: {output_path} (Hãy chạy cell xử lý embedding trước)")

print("\n" + "=" * 80)
print("ĐÃ IN XONG TOÀN BỘ DANH SÁCH FILE JSON!")
print("=" * 80)

TÌM THẤY 0 FILE JSON ĐỂ SO SÁNH

ĐÃ IN XONG TOÀN BỘ DANH SÁCH FILE JSON!
